# Knowledge Loom → FORRT Nanopublications (Single Record)

Convert one TIB Knowledge Loom record into FORRT nanopublications
(Claims, Study, Outcomes) with full dtreg metadata.

## How to run
1. Set `LOOM_RESOURCE_ID` in the Configuration cell
2. Run all cells top to bottom
3. Preview each TriG file before publishing
4. Mark the record as DONE in `loom_records.txt` when finished

For batch processing, use `loom2nanopub_batch.ipynb`.


## 1. Setup

In [36]:
import json
import re
import urllib.parse
import urllib.request
from datetime import datetime, timezone
from dataclasses import dataclass
from pathlib import Path

from rdflib import Dataset, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD, FOAF
from nanopub import load_profile

In [37]:
# Configuration
LOOM_RESOURCE_ID = "vjea9aobg7"  # From knowledgeloom.tib.eu/resource/<ID>

# Auto-lookup GitLab repo from loom_records.txt mapping
LOOM_GITLAB_REPO = ""
with open("../loom_records.txt") as _f:
    for _line in _f:
        if _line.strip().startswith(LOOM_RESOURCE_ID):
            _parts = _line.split()
            if len(_parts) >= 3:
                LOOM_GITLAB_REPO = _parts[2]
            break

PROFILE_PATH = "/Users/annef/Documents/ScienceLive/annefou-profile/profile.yml"
NP_CMD = "/Users/annef/Documents/ScienceLive/bin/np"
NP_SERVER = "https://np.knowledgepixels.com/"
OUTPUT_DIR = "../outputs/"

profile = load_profile(PROFILE_PATH)
AUTHOR_URI = URIRef(profile.orcid_id)  # CRITICAL: URIRef not ORCID[...]
AUTHOR_NAME = profile.name
print(f"Profile: {AUTHOR_NAME} ({AUTHOR_URI})")
print(f"Loom record: {LOOM_RESOURCE_ID}")
print(f"GitLab repo: {LOOM_GITLAB_REPO or '(none — dtreg parsing will be skipped)'}")
assert 'orcid.org/orcid.org' not in str(AUTHOR_URI), "Double ORCID!"


Profile: Anne Fouilloux (https://orcid.org/0000-0002-1784-2920)


In [38]:
# Namespaces
PUBLISHER = "https://sciencelive4all.org/"
TEMP_NP = Namespace("https://w3id.org/sciencelive/np")
NP = Namespace("http://www.nanopub.org/nschema#")
DCT = Namespace("http://purl.org/dc/terms/")
NT = Namespace("https://w3id.org/np/o/ntemplate/")
NPX = Namespace("http://purl.org/nanopub/x/")
PROV = Namespace("http://www.w3.org/ns/prov#")
SCHEMA = Namespace("http://schema.org/")
SCIENCELIVE = Namespace("https://w3id.org/sciencelive/o/terms/")

FORRT_CLAIM_TEMPLATE = URIRef("https://w3id.org/np/RAu5uTahAxc0OLBB3vaGwK3OQDDZV7QuWtDlBk0Ea3bco")
FORRT_KL_STUDY_TEMPLATE = URIRef("https://w3id.org/np/RALIq4JelUP-q9BuWONcKMJ87B5n59ppcwhQjl-1dheO4")
FORRT_KL_OUTCOME_TEMPLATE = URIRef("https://w3id.org/np/RAw3XdUhxQJfKBaU-cQhV6c7au4rLd5CSUdbMKTS_FB8g")
PROV_TEMPLATE = URIRef("https://w3id.org/np/RA7lSq6MuK_TIC6JMSHvLtee3lpLoZDOqLJCLXevnrPoU")
PUBINFO_TEMPLATE_1 = URIRef("https://w3id.org/np/RACJ58Gvyn91LqCKIO9zu1eijDQIeEff28iyDrJgjSJF8")
PUBINFO_TEMPLATE_2 = URIRef("https://w3id.org/np/RAukAcWHRDlkqxk7H2XNSegc1WnHI569INvNr-xdptDGI")

KL_API = "https://knowledgeloom.tib.eu/api/v1"
DTREG_TYPE_MAP = {
    "feeb33ad3e4440682a4d": SCIENCELIVE["dtreg-DataAnalysis"],
    "37182ecfb4474942e255": SCIENCELIVE["dtreg-DataPreprocessing"],
    "5b66cb584b974b186f37": SCIENCELIVE["dtreg-DescriptiveStatistics"],
    "b9335ce2c99ed87735a6": SCIENCELIVE["dtreg-GroupComparison"],
    "286991b26f02d58ee490": SCIENCELIVE["dtreg-RegressionAnalysis"],
    "3f64a93eef69d721518f": SCIENCELIVE["dtreg-CorrelationAnalysis"],
    "c6b413ba96ba477b5dca": SCIENCELIVE["dtreg-MultilevelAnalysis"],
    "6e3e29ce3ba5a0b9abfe": SCIENCELIVE["dtreg-ClassPrediction"],
    "c6e19df3b52ab8d855a9": SCIENCELIVE["dtreg-ClassDiscovery"],
    "5e782e67e70d0b2a022a": SCIENCELIVE["dtreg-AlgorithmEvaluation"],
    "437807f8d1a81b5138a3": SCIENCELIVE["dtreg-FactorAnalysis"]
}
print("Namespaces configured.")
# GitLab API (for dtreg JSON-LD files)
GITLAB_API = "https://gitlab.com/api/v4"
LOOM_GROUP = "TIBHannover/lki/knowledge-loom/loom-records"


Namespaces configured.


In [39]:
# Helpers
def fetch_json(url):
    with urllib.request.urlopen(urllib.request.Request(url)) as resp:
        return json.loads(resp.read().decode())

def gitlab_api_url(repo_path, endpoint):
    encoded = urllib.parse.quote(f"{LOOM_GROUP}/{repo_path}", safe="")
    return f"{GITLAB_API}/projects/{encoded}/{endpoint}"

def slugify(s):
    return re.sub(r'[^a-zA-Z0-9_-]', '-', s.lower()).strip('-')[:60]

def bind_all(ds):
    for p, n in [("this", TEMP_NP), ("sub", TEMP_NP), ("np", NP), ("dct", DCT),
                  ("nt", NT), ("npx", NPX), ("xsd", XSD), ("rdfs", RDFS),
                  ("prov", PROV), ("foaf", FOAF), ("schema", SCHEMA),
                  ("sciencelive", SCIENCELIVE)]:
        ds.bind(p, n)

def make_head(ds):
    this_np = URIRef(TEMP_NP)
    h = ds.graph(URIRef(TEMP_NP + "/Head"))
    h.add((this_np, RDF.type, NP.Nanopublication))
    h.add((this_np, NP.hasAssertion, URIRef(TEMP_NP + "/assertion")))
    h.add((this_np, NP.hasProvenance, URIRef(TEMP_NP + "/provenance")))
    h.add((this_np, NP.hasPublicationInfo, URIRef(TEMP_NP + "/pubinfo")))
    return this_np

def make_provenance(ds):
    p = ds.graph(URIRef(TEMP_NP + "/provenance"))
    p.add((URIRef(TEMP_NP + "/assertion"), PROV.wasAttributedTo, AUTHOR_URI))

def make_pubinfo(ds, this_np, label, template_uri, introduced_uri=None):
    pi = ds.graph(URIRef(TEMP_NP + "/pubinfo"))
    pi.add((AUTHOR_URI, FOAF.name, Literal(AUTHOR_NAME)))
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
    pi.add((this_np, DCT.created, Literal(now, datatype=XSD.dateTime)))
    pi.add((this_np, DCT.creator, AUTHOR_URI))
    pi.add((this_np, DCT.license, URIRef("https://creativecommons.org/licenses/by/4.0/")))
    pi.add((this_np, NPX.wasCreatedAt, URIRef(PUBLISHER)))
    pi.add((this_np, RDFS.label, Literal(label)))
    pi.add((this_np, NT.wasCreatedFromTemplate, template_uri))
    pi.add((this_np, NT.wasCreatedFromProvenanceTemplate, PROV_TEMPLATE))
    pi.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_1))
    pi.add((this_np, NT.wasCreatedFromPubinfoTemplate, PUBINFO_TEMPLATE_2))
    if introduced_uri:
        pi.add((this_np, NPX.introduces, introduced_uri))

def sign_and_publish(trig_file, resource_suffix=None):
    """Sign and publish. Returns (nanopub_uri, resource_uri)."""
    from nanopub import Nanopub, NanopubConf
    conf = NanopubConf(profile=profile)
    np_obj = Nanopub(rdf=trig_file, conf=conf)
    np_obj.sign()
    
    # Store signed version for inspection
    signed_path = trig_file.with_suffix(".signed.trig")
    np_obj.store(signed_path)
    
    np_obj.publish()
    np_uri = np_obj.source_uri
    
    # Build the resource URI from the signed file
    resource_uri = None
    if resource_suffix:
        # The sub: namespace in signed file is np_uri + "/"
        # So resource is np_uri + "/" + suffix
        # But np_uri from library may be short form, get full from signed file
        content = signed_path.read_text()
        import re
        match = re.search(r"prefix sub\d*: <([^>]+)>", content)
        if match:
            resource_uri = match.group(1) + resource_suffix
    
    return np_uri, resource_uri

def validate_trig(trig_file):
    content = trig_file.read_text()
    assert 'orcid.org/https' not in content, f"Double ORCID in {trig_file.name}!"
    assert '10.82209/' not in content, f"Unresolvable TIB DOI in {trig_file.name}!"
    return True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Helpers defined.")

# Fetch dtreg JSON-LD files from GitLab for software/input data metadata
@dataclass
class AnalysisInfo:
    label: str = ""
    method_label: str = ""
    method_call: str = ""
    package_name: str = ""
    package_version: str = ""
    runtime_name: str = ""
    runtime_version: str = ""
    input_label: str = ""
    input_source_url: str = ""
    input_rows: int = 0
    input_cols: int = 0
    json_file: str = ""

def get_prop(obj, suffix):
    """Get a property from a dtreg JSON-LD object by suffix (after #)."""
    if not isinstance(obj, dict): return None
    for key, val in obj.items():
        if key.endswith(f"#{suffix}") or key == suffix: return val
    return None

def parse_single_step(step, filename):
    """Extract software/input metadata from one analysis step."""
    info = AnalysisInfo(json_file=filename)
    executes = get_prop(step, "executes")
    if isinstance(executes, dict):
        info.method_label = get_prop(executes, "label") or ""
        info.method_call = get_prop(executes, "is_implemented_by") or ""
        part_of = get_prop(executes, "part_of")
        if isinstance(part_of, dict):
            info.package_name = get_prop(part_of, "label") or ""
            info.package_version = get_prop(part_of, "version_info") or ""
            runtime = get_prop(part_of, "part_of")
            if isinstance(runtime, dict):
                info.runtime_name = get_prop(runtime, "label") or ""
                info.runtime_version = get_prop(runtime, "version_info") or ""
    has_input = get_prop(step, "has_input")
    # has_input can be a dict or list of dicts
    if isinstance(has_input, list):
        has_input = has_input[0] if has_input else None
    if isinstance(has_input, dict):
        info.input_label = get_prop(has_input, "label") or ""
        info.input_source_url = get_prop(has_input, "source_url") or ""
        chars = get_prop(has_input, "has_characteristic")
        if isinstance(chars, dict):
            info.input_rows = get_prop(chars, "number_of_rows") or 0
            info.input_cols = get_prop(chars, "number_of_columns") or 0
    method_str = f"{info.package_name}::{info.method_label}" if info.package_name else info.method_label
    info.label = method_str or filename
    return info if (info.method_label or info.input_label) else None

@dataclass
class StepOutputInfo:
    """Output data for a single analysis step (maps to one KL statement)."""
    step_label: str = ""
    target_variable: str = ""
    output_label: str = ""
    output_columns: list = None
    output_values: dict = None  # {col_name: value} for first row
    output_rows: int = 0
    output_cols: int = 0
    viz_url: str = ""
    input_label: str = ""
    input_source_url: str = ""

    def __post_init__(self):
        if self.output_columns is None: self.output_columns = []
        if self.output_values is None: self.output_values = {}

def extract_step_output(step):
    """Extract output/result data from a dtreg analysis step."""
    info = StepOutputInfo()
    info.step_label = get_prop(step, "label") or ""
    # Input data
    has_input = get_prop(step, "has_input")
    if isinstance(has_input, list): has_input = has_input[0] if has_input else None
    if isinstance(has_input, dict):
        info.input_label = get_prop(has_input, "label") or ""
        info.input_source_url = get_prop(has_input, "source_url") or ""
    targets = get_prop(step, "targets")
    if isinstance(targets, dict):
        info.target_variable = get_prop(targets, "label") or ""
    elif isinstance(targets, list) and targets:
        info.target_variable = ", ".join(get_prop(t, "label") or "" for t in targets if isinstance(t, dict))
    has_output = get_prop(step, "has_output")
    if isinstance(has_output, dict):
        info.output_label = get_prop(has_output, "label") or ""
        # Column names
        parts = get_prop(has_output, "has_part")
        if isinstance(parts, list):
            info.output_columns = [get_prop(p, "label") for p in parts if isinstance(p, dict)]
        # Dimensions
        chars = get_prop(has_output, "has_characteristic")
        if isinstance(chars, dict):
            info.output_rows = get_prop(chars, "number_of_rows") or 0
            info.output_cols = get_prop(chars, "number_of_columns") or 0
        # Visualization URL
        expr = get_prop(has_output, "has_expression")
        if isinstance(expr, dict):
            info.viz_url = get_prop(expr, "source_url") or ""
        # Actual result values from first row of source_table
        src_table = get_prop(has_output, "source_table")
        if isinstance(src_table, dict):
            cols = src_table.get("columns", get_prop(src_table, "columns") or [])
            rows = src_table.get("rows", get_prop(src_table, "rows") or [])
            col_titles = []
            for c in (cols if isinstance(cols, list) else []):
                title = c.get("col_titles", get_prop(c, "col_titles") or get_prop(c, "titles") or "")
                col_titles.append(title)
            if isinstance(rows, list) and rows:
                cells = rows[0].get("cells", get_prop(rows[0], "cells") or [])
                if isinstance(cells, list):
                    for ci, cell in enumerate(cells):
                        v = get_prop(cell, "value") or get_prop(cell, "primary_value") or ""
                        col_name = col_titles[ci] if ci < len(col_titles) else info.output_columns[ci] if ci < len(info.output_columns) else f"col{ci}"
                        info.output_values[col_name] = str(v)
    return info

def extract_analysis_steps(dtreg_data, filename):
    """Extract all analysis steps from a dtreg JSON-LD file. Returns list of AnalysisInfo."""
    results = []
    has_part = get_prop(dtreg_data, "has_part")
    # has_part can be a single dict or a list of steps
    if isinstance(has_part, dict):
        steps = [has_part]
    elif isinstance(has_part, list):
        steps = has_part
    else:
        steps = [dtreg_data]
    for step in steps:
        if not isinstance(step, dict):
            continue
        info = parse_single_step(step, filename)
        if info:
            results.append(info)
    return results


Helpers defined.


## 2. Fetch Knowledge Loom data

In [40]:
kl_data = fetch_json(f"{KL_API}/articles/get_article_by_id/?id={LOOM_RESOURCE_ID}")
article = kl_data.get("article", kl_data)
statements = kl_data.get("statements", [])
datasets = kl_data.get("basises", [])
kl_url = f"https://knowledgeloom.tib.eu/resource/{LOOM_RESOURCE_ID}"

# Source DOI: use dataset/basis DOI (= the source publication)
source_doi = None
for d in datasets:
    if d.get("id", "").startswith("http"):
        source_doi = d["id"]
        break

print(f"Title: {article['name']}")
print(f"Source DOI: {source_doi}")
print(f"KL URL: {kl_url}")
print(f"\nStatements ({len(statements)}):")
for s in statements:
    print(f"  - {s['label'][:80]}")
    print(f"    Type: {s.get('type', {}).get('name', '?')}")

Title: Determinants of Bactrocera oleae abundance in olive groves
Source DOI: https://doi.org/10.1007/s10340-022-01489-1
KL URL: https://knowledgeloom.tib.eu/resource/vjea9aobg7

Statements (2):
  - Based on model results, there was a significant negative relationship between th
    Type: Regression analysis
  - Based on model results, a higher percentage of olive groves in the landscape was
    Type: Regression analysis


In [ ]:
analyses = []
script_url = ""

if LOOM_GITLAB_REPO:
    gitlab_url = f"https://gitlab.com/{LOOM_GROUP}/{LOOM_GITLAB_REPO}"
    tree = fetch_json(gitlab_api_url(LOOM_GITLAB_REPO, "repository/tree?per_page=100&ref=main"))
    json_files = [f["name"] for f in tree if f["name"].endswith(".json")]
    print(f"GitLab repo: {gitlab_url}")
    print(f"Found {len(json_files)} JSON files: {json_files}")
    for jf in json_files:
        encoded_jf = urllib.parse.quote(jf, safe="")
        raw_url = gitlab_api_url(LOOM_GITLAB_REPO, f"repository/files/{encoded_jf}/raw?ref=main")
        dtreg = fetch_json(raw_url)
        # Script URL from top-level is_implemented_by
        impl = get_prop(dtreg, "is_implemented_by")
        if isinstance(impl, str) and impl.startswith("http"):
            script_url = impl
        infos = extract_analysis_steps(dtreg, jf)
        for info in infos:
            analyses.append(info)
            print(f"  {jf}: {info.label}")
            if info.package_name:
                print(f"    Software: {info.package_name} {info.package_version}")
            if info.runtime_name:
                print(f"    Runtime: {info.runtime_name} {info.runtime_version}")
            if info.input_label:
                print(f"    Input: {info.input_label}")
    if script_url:
        print(f"  Script: {script_url}")
else:
    print("No LOOM_GITLAB_REPO set — skipping dtreg parsing (software/input data will be empty).")

# Extract input dataset URLs from metadata.json file_mapping
input_data_urls = []
if LOOM_GITLAB_REPO:
    try:
        encoded_meta = urllib.parse.quote("utils/metadata.json", safe="")
        meta_url = gitlab_api_url(LOOM_GITLAB_REPO, f"repository/files/{encoded_meta}/raw?ref=main")
        _metadata = fetch_json(meta_url)
        _fm = _metadata.get("file_mapping", {})
        data_exts = {"csv", "xlsx", "tsv", "rds", "rda", "dat", "sav", "txt"}
        for fname, finfo in _fm.items():
            ext = fname.rsplit(".", 1)[-1].lower() if "." in fname else ""
            if ext in data_exts and finfo.get("resource_url"):
                input_data_urls.append((fname, finfo["resource_url"]))
        if input_data_urls:
            print(f"\nInput datasets ({len(input_data_urls)}):")
            for fname, url in input_data_urls[:5]:
                print(f"  {fname}: {url[:70]}")
    except Exception as e:
        print(f"Warning: could not extract data URLs: {e}")

# Build per-statement mapping: KL statement label -> dtreg step output data
stmt_output_map = {}  # statement_label -> StepOutputInfo
if LOOM_GITLAB_REPO:
    try:
        encoded_meta = urllib.parse.quote("utils/metadata.json", safe="")
        meta_url = gitlab_api_url(LOOM_GITLAB_REPO, f"repository/files/{encoded_meta}/raw?ref=main")
        metadata = fetch_json(meta_url)
        meta_stmts = metadata.get("statements", {})
        file_mapping = metadata.get("file_mapping", {})
        # Build label -> actual JSON filename
        label_to_json = {}
        for sid, sinfo in meta_stmts.items():
            label = sinfo.get("label", "")
            json_orig = sinfo.get("json_file_name", "")
            mapped = file_mapping.get(json_orig, {}).get("mapped_name", "")
            if label and mapped:
                label_to_json[label] = mapped
            # Also store stripped version for fuzzy match
            if label.strip() and mapped:
                label_to_json[label.strip()] = mapped
        # For each statement, fetch its JSON and extract the last step's output
        # (last step is typically the main analysis result)
        for label, json_fname in label_to_json.items():
            encoded_jf = urllib.parse.quote(json_fname, safe="")
            raw_url = gitlab_api_url(LOOM_GITLAB_REPO, f"repository/files/{encoded_jf}/raw?ref=main")
            dtreg = fetch_json(raw_url)
            has_part = get_prop(dtreg, "has_part")
            steps = has_part if isinstance(has_part, list) else [has_part] if isinstance(has_part, dict) else []
            # Use the last step (main result) for output
            for step in reversed(steps):
                if isinstance(step, dict):
                    output_info = extract_step_output(step)
                    if output_info.output_values or output_info.output_label or output_info.viz_url:
                        stmt_output_map[label] = output_info
                        break
        print(f"\nStatement-to-output mapping: {len(stmt_output_map)}/{len(label_to_json)} statements have output data")
        for lbl, out in list(stmt_output_map.items())[:3]:
            vals_str = ", ".join(f"{k}={v}" for k, v in list(out.output_values.items())[:3])
            print(f"  {lbl[:60]}...")
            print(f"    → {out.output_label or out.step_label}: {vals_str}")
    except Exception as e:
        print(f"Warning: could not build statement-output mapping: {e}")


## 3. Generate and publish FORRT Claims

One claim per Knowledge Loom statement. **Review the preview before publishing.**

In [41]:
def create_claim(stmt):
    ds = Dataset()
    bind_all(ds)
    this_np = make_head(ds)
    claim_id = slugify(f"kl-claim-{stmt['statement_id']}")
    claim_uri = URIRef(str(TEMP_NP) + "/" + claim_id)
    aida_sentence = stmt["label"]
    aida_uri = URIRef(f"http://purl.org/aida/{urllib.parse.quote(aida_sentence, safe='')}")
    a = ds.graph(URIRef(TEMP_NP + "/assertion"))
    a.add((claim_uri, RDF.type, SCIENCELIVE["FORRT-Claim"]))
    # Auto-map KL analysis type to FORRT claim type
    kl_type = stmt.get("type", {}).get("name", "").lower()
    if any(t in kl_type for t in ["group comparison", "regression", "correlation", "multilevel"]):
        forrt_type = SCIENCELIVE["statistical_significance-FORRT-Claim"]
    elif "algorithm evaluation" in kl_type:
        forrt_type = SCIENCELIVE["model_performance-FORRT-Claim"]
    elif any(t in kl_type for t in ["descriptive", "factor"]):
        forrt_type = SCIENCELIVE["descriptive_pattern-FORRT-Claim"]
    elif "preprocessing" in kl_type:
        forrt_type = SCIENCELIVE["data_quality-FORRT-Claim"]
    else:
        forrt_type = SCIENCELIVE["statistical_significance-FORRT-Claim"]
    a.add((claim_uri, RDF.type, forrt_type))
    a.add((claim_uri, RDFS.label, Literal(aida_sentence)))
    a.add((claim_uri, SCIENCELIVE["asAidaStatement"], aida_uri))
    if source_doi:
        a.add((claim_uri, DCT.source, URIRef(source_doi)))
    # Input data source from dtreg analyses
    for ai in analyses:
        if ai.input_label:
            desc = ai.input_label
            if ai.input_rows and ai.input_cols: desc += f" ({ai.input_rows} rows x {ai.input_cols} columns)"
            a.add((claim_uri, SCIENCELIVE["hasDataSource"], Literal(desc)))
            break  # Use first analysis input as representative data source
    make_provenance(ds)
    label = f"FORRT Claim: {aida_sentence[:80]}{'...' if len(aida_sentence) > 80 else ''}"
    make_pubinfo(ds, this_np, label, FORRT_CLAIM_TEMPLATE, claim_uri)
    return ds, label

claim_files = []
for i, stmt in enumerate(statements):
    ds, label = create_claim(stmt)
    f = Path(OUTPUT_DIR) / f"{LOOM_RESOURCE_ID}-claim-{i+1}.trig"
    ds.serialize(destination=str(f), format='trig')
    validate_trig(f)
    claim_files.append(f)
    print(f"  {f.name} — {label}")
print(f"\nGenerated {len(claim_files)} claims.")

  vjea9aobg7-claim-1.trig — FORRT Claim: Based on model results, there was a significant negative relationship between th...
  vjea9aobg7-claim-2.trig — FORRT Claim: Based on model results, a higher percentage of olive groves in the landscape was...

Generated 2 claims.


In [42]:
# Preview first claim
if claim_files:
    print(claim_files[0].read_text())

@prefix dct: <http://purl.org/dc/terms/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix np: <http://www.nanopub.org/nschema#> .
@prefix npx: <http://purl.org/nanopub/x/> .
@prefix nt: <https://w3id.org/np/o/ntemplate/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sciencelive: <https://w3id.org/sciencelive/o/terms/> .
@prefix sub: <https://w3id.org/sciencelive/np> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://w3id.org/sciencelive/np/assertion> {
    <https://w3id.org/sciencelive/np/kl-claim-d9h0kie3p9> a sciencelive:FORRT-Claim,
            sciencelive:statistical_significance-FORRT-Claim ;
        rdfs:label "Based on model results, there was a significant negative relationship between the Shannon diversity index and B. oleae abundance." ;
        dct:source <https://doi.org/10.1007/s10340-022-01489-1> ;
        sciencelive:asAidaStatement <http://purl.org/aida/Based%20on%20model%20results%2C%20t

In [44]:
# PUBLISH CLAIMS — run only when satisfied with the preview
claim_uris = []  # Claim RESOURCE URIs (for targetsClaim in Study)
for idx, f in enumerate(claim_files):
    suffix = slugify(f"kl-claim-{statements[idx]['statement_id']}")
    np_uri, resource_uri = sign_and_publish(f, resource_suffix=suffix)
    claim_uris.append(resource_uri)
    print(f"Published: {np_uri}")
    print(f"  Claim resource: {resource_uri}")
print(f"{len(claim_uris)} claims published.")
print("Use these claim_uris for targetsClaim in the Study.")

Published: https://w3id.org/sciencelive/np/RAd81Aae_Buc4blS-RseBVlFybfD32gxisppiho-INgWg
  Claim resource: https://w3id.org/sciencelive/np/RAd81Aae_Buc4blS-RseBVlFybfD32gxisppiho-INgWg/kl-claim-d9h0kie3p9
Published: https://w3id.org/sciencelive/np/RAgqt-UhDLp-Zl-1zBTrFyCOHj_bMQWnoaYRR3eTGdKGw
  Claim resource: https://w3id.org/sciencelive/np/RAgqt-UhDLp-Zl-1zBTrFyCOHj_bMQWnoaYRR3eTGdKGw/kl-claim-9p3urx5m79
2 claims published.
Use these claim_uris for targetsClaim in the Study.


## 4. Generate and publish FORRT-KL Replication Study

Links to all published Claims via `targetsClaim`. **Review before publishing.**

In [45]:
def create_study():
    ds = Dataset()
    bind_all(ds)
    this_np = make_head(ds)
    study_id = slugify(f"kl-study-{LOOM_RESOURCE_ID}")
    study_uri = URIRef(str(TEMP_NP) + "/" + study_id)
    a = ds.graph(URIRef(TEMP_NP + "/assertion"))
    a.add((study_uri, RDF.type, SCIENCELIVE["FORRT-Replication-Study"]))
    a.add((study_uri, RDF.type, SCIENCELIVE["Reproduction-Study"]))
    label = f"Replication study: {article.get('name', LOOM_RESOURCE_ID)}"
    a.add((study_uri, RDFS.label, Literal(label)))
    # targetsClaim — link to all published claims
    for claim_uri in claim_uris:
        a.add((study_uri, SCIENCELIVE["targetsClaim"], URIRef(claim_uri)))
    # Scope
    abstract = article.get("abstract", "")
    scope = f"Reproduction of analyses from Knowledge Loom record: {article.get('name', LOOM_RESOURCE_ID)}"
    if abstract:
        scope += f". {abstract}"
    a.add((study_uri, SCIENCELIVE["hasScopeDescription"], Literal(scope)))
    # Methodology
    types = set(s.get("type", {}).get("name", "") for s in statements if s.get("type", {}).get("name"))
    method = f"Analysis types: {', '.join(sorted(types))}." if types else ""
    method += " Machine-readable descriptions generated using dtreg and published in the TIB Knowledge Loom."
    a.add((study_uri, SCIENCELIVE["hasMethodologyDescription"], Literal(method.strip())))
    # Software and input data from dtreg
    if analyses:
        methods, packages, runtimes, input_descs, input_urls = set(), set(), set(), set(), set()
        for ai in analyses:
            if ai.method_call: methods.add(ai.method_call.split("\n")[0].strip())
            elif ai.method_label and ai.package_name: methods.add(f"{ai.package_name}::{ai.method_label}()")
            if ai.package_name: packages.add(f"{ai.package_name} {ai.package_version}".strip())
            if ai.runtime_name: runtimes.add(f"{ai.runtime_name} {ai.runtime_version}".strip())
            if ai.input_label:
                desc = ai.input_label
                if ai.input_rows and ai.input_cols: desc += f" ({ai.input_rows} rows x {ai.input_cols} columns)"
                input_descs.add(desc)
            if ai.input_source_url: input_urls.add(ai.input_source_url)
        if methods: a.add((study_uri, SCIENCELIVE["executesMethod"], Literal("; ".join(sorted(methods)))))
        if packages: a.add((study_uri, SCIENCELIVE["usesSoftwarePackage"], Literal("; ".join(sorted(packages)))))
        if runtimes: a.add((study_uri, SCIENCELIVE["hasRuntimeEnvironment"], Literal("; ".join(sorted(runtimes)))))
        if input_descs: a.add((study_uri, SCIENCELIVE["hasInputDataDescription"], Literal("; ".join(sorted(input_descs)))))
        for url in sorted(input_urls): a.add((study_uri, SCIENCELIVE["hasInputDataSource"], URIRef(url)))
    # Input dataset URLs from metadata.json
    for fname, url in input_data_urls:
        a.add((study_uri, SCIENCELIVE["hasInputDataset"], URIRef(url)))
    if script_url: a.add((study_uri, SCIENCELIVE["hasAnalysisScript"], URIRef(script_url)))
    # KL link
    a.add((study_uri, SCIENCELIVE["hasLoomRecord"], URIRef(kl_url)))
    make_provenance(ds)
    make_pubinfo(ds, this_np, label, FORRT_KL_STUDY_TEMPLATE, study_uri)
    return ds, label

ds, label = create_study()
study_file = Path(OUTPUT_DIR) / f"{LOOM_RESOURCE_ID}-study.trig"
ds.serialize(destination=str(study_file), format='trig')
validate_trig(study_file)
print(f"Generated: {study_file.name} — {label}")
print(f"Links to {len(claim_uris)} claims.")

Generated: vjea9aobg7-study.trig — Replication study: Determinants of Bactrocera oleae abundance in olive groves
Links to 2 claims.


In [46]:
# Preview study
print(study_file.read_text())

@prefix dct: <http://purl.org/dc/terms/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix np: <http://www.nanopub.org/nschema#> .
@prefix npx: <http://purl.org/nanopub/x/> .
@prefix nt: <https://w3id.org/np/o/ntemplate/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sciencelive: <https://w3id.org/sciencelive/o/terms/> .
@prefix sub: <https://w3id.org/sciencelive/np> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://w3id.org/sciencelive/np/assertion> {
    <https://w3id.org/sciencelive/np/kl-study-vjea9aobg7> a sciencelive:FORRT-Replication-Study,
            sciencelive:Reproduction-Study ;
        rdfs:label "Replication study: Determinants of Bactrocera oleae abundance in olive groves" ;
        sciencelive:hasLoomRecord <https://knowledgeloom.tib.eu/resource/vjea9aobg7> ;
        sciencelive:hasMethodologyDescription "Analysis types: Regression analysis. Machine-readable descriptions generated using 

In [47]:
# PUBLISH STUDY
study_suffix = slugify(f"kl-study-{LOOM_RESOURCE_ID}")
study_np_uri, study_uri = sign_and_publish(study_file, resource_suffix=study_suffix)
print(f"Published Study NP: {study_np_uri}")
print(f"Study resource: {study_uri}")

Published Study NP: https://w3id.org/sciencelive/np/RACSKwvXe5B441qTr7AuTzmT74g03VRuERc-31egcScZI
Study resource: https://w3id.org/sciencelive/np/RACSKwvXe5B441qTr7AuTzmT74g03VRuERc-31egcScZI/kl-study-vjea9aobg7


## 5. Generate and publish FORRT-KL Replication Outcomes

One per statement, links to Study via `isOutcomeOf`. **Review before publishing.**

In [48]:
def create_outcome(stmt, index):
    ds = Dataset()
    bind_all(ds)
    this_np = make_head(ds)
    outcome_id = slugify(f"kl-outcome-{LOOM_RESOURCE_ID}-{index}")
    outcome_uri = URIRef(str(TEMP_NP) + "/" + outcome_id)
    stmt_label = stmt["label"]
    type_info = stmt.get("type", {})
    type_name = type_info.get("name", "analysis")
    label = f"Outcome ({type_name}): {stmt_label[:60]}{'...' if len(stmt_label) > 60 else ''}"
    today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    a = ds.graph(URIRef(TEMP_NP + "/assertion"))
    a.add((outcome_uri, RDF.type, SCIENCELIVE["FORRT-Replication-Outcome"]))
    a.add((outcome_uri, RDFS.label, Literal(label)))
    a.add((outcome_uri, SCIENCELIVE["isOutcomeOf"], URIRef(study_uri)))
    a.add((outcome_uri, SCIENCELIVE["hasOutcomeRepository"], URIRef(kl_url)))
    a.add((outcome_uri, SCHEMA.endDate, Literal(today, datatype=XSD.date)))
    a.add((outcome_uri, SCIENCELIVE["hasValidationStatus"], SCIENCELIVE["Validated"]))
    a.add((outcome_uri, SCIENCELIVE["hasConfidenceLevel"], SCIENCELIVE["HighConfidence"]))
    a.add((outcome_uri, SCIENCELIVE["hasConclusionDescription"],
           Literal(f"Knowledge Loom verified: {stmt_label}")))
    # Enrich with output data from dtreg (per-statement mapping)
    output_info = stmt_output_map.get(stmt_label) or stmt_output_map.get(stmt_label.strip())
    if output_info:
        if output_info.target_variable:
            a.add((outcome_uri, SCIENCELIVE["hasTargetVariable"], Literal(output_info.target_variable)))
        if output_info.output_label:
            a.add((outcome_uri, SCIENCELIVE["hasOutputDescription"], Literal(output_info.output_label)))
        if output_info.output_values:
            vals_str = "; ".join(f"{k} = {v}" for k, v in output_info.output_values.items())
            a.add((outcome_uri, SCIENCELIVE["hasResultValues"], Literal(vals_str)))
        if output_info.output_columns:
            a.add((outcome_uri, SCIENCELIVE["hasResultMetrics"], Literal(", ".join(output_info.output_columns))))
        if output_info.viz_url:
            a.add((outcome_uri, SCIENCELIVE["hasResultVisualization"], URIRef(output_info.viz_url)))
        if output_info.step_label:
            a.add((outcome_uri, SCIENCELIVE["hasAnalysisDescription"], Literal(output_info.step_label)))
        evidence = f"Reproduced {type_name}: {output_info.step_label or stmt_label}."
        if output_info.output_values:
            top_vals = "; ".join(f"{k}={v}" for k, v in list(output_info.output_values.items())[:5])
            evidence += f" Result: {top_vals}."
    else:
        evidence = f"Machine-readable analysis proof ({type_name}) published in the TIB Knowledge Loom."
    a.add((outcome_uri, SCIENCELIVE["hasEvidenceDescription"], Literal(evidence)))
    # Input data source for this specific outcome
    if output_info:
        if output_info.input_label:
            a.add((outcome_uri, SCIENCELIVE["hasInputDataDescription"], Literal(output_info.input_label)))
        if output_info.input_source_url:
            a.add((outcome_uri, SCIENCELIVE["hasInputDataSource"], URIRef(output_info.input_source_url)))
    a.add((outcome_uri, SCIENCELIVE["hasMachineReadableProof"], URIRef(kl_url)))
    type_hash = type_info.get("type_id", "")
    if type_hash in DTREG_TYPE_MAP:
        a.add((outcome_uri, SCIENCELIVE["hasAnalysisType"], DTREG_TYPE_MAP[type_hash]))
    make_provenance(ds)
    make_pubinfo(ds, this_np, label, FORRT_KL_OUTCOME_TEMPLATE, outcome_uri)
    return ds, label

outcome_files = []
for i, stmt in enumerate(statements):
    ds, label = create_outcome(stmt, i+1)
    f = Path(OUTPUT_DIR) / f"{LOOM_RESOURCE_ID}-outcome-{i+1}.trig"
    ds.serialize(destination=str(f), format='trig')
    validate_trig(f)
    outcome_files.append(f)
    print(f"  {f.name} — {label}")
print(f"\nGenerated {len(outcome_files)} outcomes. All link to study: {study_uri}")

  vjea9aobg7-outcome-1.trig — Outcome (Regression analysis): Based on model results, there was a significant negative rel...
  vjea9aobg7-outcome-2.trig — Outcome (Regression analysis): Based on model results, a higher percentage of olive groves ...

Generated 2 outcomes. All link to study: https://w3id.org/sciencelive/np/RACSKwvXe5B441qTr7AuTzmT74g03VRuERc-31egcScZI/kl-study-vjea9aobg7


In [49]:
# Preview first outcome
if outcome_files:
    print(outcome_files[0].read_text())

@prefix dct: <http://purl.org/dc/terms/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix np: <http://www.nanopub.org/nschema#> .
@prefix npx: <http://purl.org/nanopub/x/> .
@prefix nt: <https://w3id.org/np/o/ntemplate/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix schema1: <http://schema.org/> .
@prefix sciencelive: <https://w3id.org/sciencelive/o/terms/> .
@prefix sub: <https://w3id.org/sciencelive/np> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://w3id.org/sciencelive/np/assertion> {
    <https://w3id.org/sciencelive/np/kl-outcome-vjea9aobg7-1> a sciencelive:FORRT-Replication-Outcome ;
        rdfs:label "Outcome (Regression analysis): Based on model results, there was a significant negative rel..." ;
        schema1:endDate "2026-04-02"^^xsd:date ;
        sciencelive:hasAnalysisType sciencelive:dtreg-RegressionAnalysis ;
        sciencelive:hasConclusionDescription "Knowledge Loom verified: Bas

In [51]:
# PUBLISH OUTCOMES
outcome_uris = []
for idx, f in enumerate(outcome_files):
    suffix = slugify(f"kl-outcome-{LOOM_RESOURCE_ID}-{idx+1}")
    np_uri, resource_uri = sign_and_publish(f, resource_suffix=suffix)
    outcome_uris.append(resource_uri)
    print(f"Published: {np_uri}")
    print(f"  Outcome resource: {resource_uri}")
print(f"{len(outcome_uris)} outcomes published.")

Published: https://w3id.org/sciencelive/np/RASjmKLYnaIZbBqSLkhkR-oaJL6V2cGq_8rdSAtUjBUq8
  Outcome resource: https://w3id.org/sciencelive/np/RASjmKLYnaIZbBqSLkhkR-oaJL6V2cGq_8rdSAtUjBUq8/kl-outcome-vjea9aobg7-1
Published: https://w3id.org/sciencelive/np/RARJlOYOtb-j26tm4iO-pxfZ6GCQlmGVHeZlhLHiunIEc
  Outcome resource: https://w3id.org/sciencelive/np/RARJlOYOtb-j26tm4iO-pxfZ6GCQlmGVHeZlhLHiunIEc/kl-outcome-vjea9aobg7-2
2 outcomes published.


## 6. Summary

In [ ]:
print(f"Knowledge Loom record: {kl_url}")
print(f"Article: {article['name']}")
print(f"\nPublished nanopublications:")
print(f"\n  Claims ({len(claim_uris)}):")
for i, uri in enumerate(claim_uris):
    print(f"    {i+1}. {uri}")
print(f"\n  Study:")
print(f"    {study_uri}")
print(f"\n  Outcomes ({len(outcome_uris)}):")
for i, uri in enumerate(outcome_uris):
    print(f"    {i+1}. {uri}")
print(f"\nTotal: {len(claim_uris) + 1 + len(outcome_uris)} nanopublications")